# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library, based on the Croissant schema. You will learn to load, inspect, and analyze the data using only entity `@id` references as per the schema specification.

### Dataset Source
The dataset Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the FAIR² dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", getattr(metadata, 'name', None))
print("Description: ", getattr(metadata, 'description', None))


## 2. Data Overview
Review and inspect available record sets in the dataset and their fields. All references use the entity `@id` as per Croissant. Here we print all record set `@id`s and give a preview of their structure.

In [ ]:
# List all record sets' @id
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
    # Handle both list-of-objects and single-object
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    print("Available record sets (@id):")
    for rs in record_sets:
        print(" -", getattr(rs, '@id', getattr(rs, 'id', rs)))
else:
    print("No record sets found in 'recordSet'. Trying fallback via 'ds.record_sets'...")
    record_sets = list(dataset.record_sets)
    for rs in record_sets:
        print(" -", rs)

# Preview the structure of the FIRST record set (using @id)
if record_sets:
    record_set_id = getattr(record_sets[0], '@id', record_sets[0])
    print(f"\nPreviewing structure for record set '@id': {record_set_id}\n")
    example_iter = dataset.records(record_set=record_set_id)
    n = 3
    for i, rec in enumerate(example_iter):
        print(f"Record {i+1}:", rec)
        if i+1 >= n:
            break


## 3. Data Extraction

Load all records for each record set into pandas DataFrames for further manipulation. All indexing is by record set `@id`.

In [ ]:
# Gather all record set @ids
record_set_ids = []
if hasattr(metadata, 'recordSet'):
    all_rs = metadata.recordSet
    if not isinstance(all_rs, list):
        all_rs = [all_rs]
    record_set_ids = [getattr(rs, '@id', getattr(rs, 'id', rs)) for rs in all_rs]
else:
    # fallback to dataset.record_sets
    record_set_ids = list(dataset.record_sets)

print("Record set @ids to load:", record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    # Load all records into DataFrame
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(records)} records for {rs_id}")
    else:
        print(f"No records for {rs_id}")

# Display columns and first few rows for the first available DataFrame
if dataframes:
    first_rs = next(iter(dataframes.keys()))
    print(f"\nColumns present in record set {first_rs}:\n", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply some basic data processing: filter based on a numeric field, normalize it, and group data. All columns refer to their field `@id` as column names.

Please inspect the DataFrame's columns above to choose a suitable numeric field and a group (categorical) field -- typically, the mass spectrometry dataset includes numeric measurements such as 'cr:MW' (molecular weight) and categorical attributes like 'cr:description'.

In [ ]:
# Example field @ids -- update these if the fields differ for your specific dataset
example_rs = None
if dataframes:
    example_rs = next(iter(dataframes.keys()))
    df = dataframes[example_rs]
    # Heuristically select numeric and group fields
    # If known field @ids, can use e.g. 'cr:MW', 'cr:accession', etc.
    numeric_candidate = None
    group_candidate = None
    for col in df.columns:
        # Look for common patterns in biological tables
        if 'coverage' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower() or 'count' in col.lower():
            # Only assign first encoutered
            if numeric_candidate is None:
                numeric_candidate = col
        if 'description' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower():
            if group_candidate is None:
                group_candidate = col

    if numeric_candidate is None:
        # fallback: look for first numeric column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_candidate = col
                break
    if group_candidate is None:
        group_candidate = df.columns[0] # fallback

    print(f"Field selected for numeric analysis: {numeric_candidate}")
    print(f"Field selected for grouping: {group_candidate}")

    # Filtering step
    if numeric_candidate is not None:
        threshold = df[numeric_candidate].mean()
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > {threshold:.2f} (mean value):")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df = filtered_df.copy()
        filtered_df[numeric_candidate + '_normalized'] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
        print(f"\nNormalized {numeric_candidate} for filtered records:")
        display(filtered_df[[numeric_candidate, numeric_candidate + '_normalized']].head())

        # Group by group_candidate and get mean
        if group_candidate in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean()
            print(f"\nGrouped average {numeric_candidate} by {group_candidate}:")
            display(grouped_df.head())
    else:
        print("No numeric field available for analysis.")

## 5. Visualization
Visualize distributions and relationships of numeric fields extracted from the dataset. All visual references are based on field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Plot histogram of the main numeric field
if dataframes and numeric_candidate is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_candidate].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_candidate}")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by a grouping field
if dataframes and numeric_candidate is not None and group_candidate is not None:
    # For visualization, use only top N groups by record count
    top_groups = df[group_candidate].value_counts().index[:5]
    plt.figure(figsize=(10,6))
    sns.boxplot(data=df[df[group_candidate].isin(top_groups)], x=group_candidate, y=numeric_candidate)
    plt.title(f"{numeric_candidate} by {group_candidate} (top 5 groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and performed simple EDA on the FAIR² mass spectrometry dataset using the Croissant schema and `mlcroissant` library, referencing all entities by their `@id`. This approach ensures complete traceability and reproducibility when accessing structured scientific data.

**Key takeaways:**
- All data elements are referenced by their Croissant `@id`.
- The dataset is ready for advanced statistical analysis or machine learning.
- Use the `mlcroissant` library for seamless Croissant schema compliance in FAIR data explorations.
